# 04 — Machine Learning: Patient-Level Asthma Diagnosis Prediction

**Input:** `data/processed/asthma_cleaned.csv`
**Target:** `Diagnosis` (0 = Negative, 1 = Positive) — ~48.95% positive, a near-balanced class split, and (per the EDA in `03_EDA_Patient.ipynb`) genuine, statistically significant signal in most features.

**Algorithms (per project spec):**
- Logistic Regression (interpretable baseline)
- Decision Tree
- Random Forest
- XGBoost
- Support Vector Machine (SVM)

All models still use `class_weight='balanced'` (or XGBoost's `scale_pos_weight`) for consistency and best practice, though with a near-50/50 split this matters far less than it would for a rare-event problem. Because the classes are balanced, accuracy is a much more trustworthy metric here than it was on the original imbalanced (~5% positive) dataset — but we still lead with ROC-AUC, precision, recall, and F1 in `05_Model_Evaluation.ipynb` for a complete picture.

This notebook only **trains and saves** the models + test split; all evaluation, comparison charts, and the final model selection happen in `05_Model_Evaluation.ipynb`.


In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

RANDOM_STATE = 42

df = pd.read_csv("../data/processed/asthma_cleaned.csv")
print("Shape:", df.shape)
df['Diagnosis'].value_counts(normalize=True).mul(100).round(2)


Shape: (2392, 50)


Diagnosis
0    51.05
1    48.95
Name: proportion, dtype: float64

## 1. Feature selection & preprocessing

In [2]:
# Drop identifier and redundant/label-duplicate columns; keep numeric-encoded features for modeling
drop_cols = ['PatientID', 'Diagnosis', 'Diagnosis_Label'] + [c for c in df.columns if c.endswith('_Label')]

feature_cols = [c for c in df.columns if c not in drop_cols and c not in ['AgeGroup', 'BMICategory']]
X = df[feature_cols].copy()
y = df['Diagnosis'].copy()

print(f"Using {len(feature_cols)} features:")
print(feature_cols)


Using 29 features:
['Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'Smoking', 'PhysicalActivity', 'DietQuality', 'SleepQuality', 'PollutionExposure', 'PollenExposure', 'DustExposure', 'PetAllergy', 'FamilyHistoryAsthma', 'HistoryOfAllergies', 'Eczema', 'HayFever', 'GastroesophagealReflux', 'LungFunctionFEV1', 'LungFunctionFVC', 'Wheezing', 'ShortnessOfBreath', 'ChestTightness', 'Coughing', 'NighttimeSymptoms', 'ExerciseInduced', 'FEV1_FVC_Ratio', 'SymptomCount', 'RiskFactorCount']


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape, " Positive rate:", y_train.mean().round(4))
print("Test shape: ", X_test.shape, " Positive rate:", y_test.mean().round(4))


Train shape: (1913, 29)  Positive rate: 0.4898
Test shape:  (479, 29)  Positive rate: 0.4885


In [4]:
# Scale features for distance/gradient-based models (Logistic Regression, SVM).
# Tree-based models (Decision Tree, Random Forest, XGBoost) don't need scaling but scaled
# inputs don't hurt them, so we keep one scaled version for convenience.
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Scaling complete.")


Scaling complete.


## 2. Handling class imbalance

This version of the dataset is close to balanced (48.95% positive), so imbalance is a much smaller concern than in the original file. `class_weight='balanced'` is still applied for consistency and as good practice, but it's not expected to be the deciding factor in model performance here — the real signal comes from the features themselves (see EDA).


In [5]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight for XGBoost: {scale_pos_weight:.2f}")


scale_pos_weight for XGBoost: 1.04


## 3. Train models

### 3.1 Logistic Regression

In [6]:
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)
print("Logistic Regression trained.")


Logistic Regression trained.


### 3.2 Decision Tree

In [7]:
dt = DecisionTreeClassifier(class_weight='balanced', max_depth=6, min_samples_leaf=10, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
print("Decision Tree trained.")


Decision Tree trained.


### 3.3 Random Forest

In [8]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
print("Random Forest trained.")


Random Forest trained.


### 3.4 XGBoost

In [9]:
xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, eval_metric='logloss',
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb.fit(X_train, y_train)
print("XGBoost trained.")


XGBoost trained.


### 3.5 Support Vector Machine (SVM)

In [10]:
svm = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_scaled, y_train)
print("SVM trained.")


SVM trained.


## 4. Quick sanity check — training accuracy (not the final metric, just a smoke test)

In [11]:
models_scaled = {'Logistic Regression': log_reg, 'SVM': svm}
models_raw = {'Decision Tree': dt, 'Random Forest': rf, 'XGBoost': xgb}

for name, m in models_scaled.items():
    print(f"{name:20s} train accuracy: {m.score(X_train_scaled, y_train):.3f}")
for name, m in models_raw.items():
    print(f"{name:20s} train accuracy: {m.score(X_train, y_train):.3f}")


Logistic Regression  train accuracy: 0.720
SVM                  train accuracy: 0.822
Decision Tree        train accuracy: 0.760


Random Forest        train accuracy: 0.845
XGBoost              train accuracy: 0.914


## 5. Save models, splits, and scaler

Everything needed for `05_Model_Evaluation.ipynb` is saved to `../models/` so evaluation can be
re-run independently without retraining.

In [12]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(log_reg, '../models/logistic_regression.pkl')
joblib.dump(dt, '../models/decision_tree.pkl')
joblib.dump(rf, '../models/random_forest.pkl')
joblib.dump(xgb, '../models/xgboost.pkl')
joblib.dump(svm, '../models/svm.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

X_train.to_csv('../models/X_train.csv', index=False)
X_test.to_csv('../models/X_test.csv', index=False)
X_train_scaled.to_csv('../models/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../models/X_test_scaled.csv', index=False)
y_train.to_csv('../models/y_train.csv', index=False)
y_test.to_csv('../models/y_test.csv', index=False)

print("All models and data splits saved to ../models/")
print(os.listdir('../models'))


All models and data splits saved to ../models/
['logistic_regression.pkl', 'X_test.csv', 'random_forest.pkl', 'X_train.csv', 'xgboost.pkl', 'svm.pkl', 'y_train.csv', 'y_test.csv', 'X_train_scaled.csv', 'X_test_scaled.csv', 'scaler.pkl', 'decision_tree.pkl']
